# AIHA scRNA-seq Drug Repurposing Pipeline
**Dataset:** GSE301528 — Bone marrow aspirate, Autoimmune Hemolytic Anemia  
**Conditions:** Diagnosis (n=5) · Remission (n=3, internal control) · Relapse/Refractory (n=4)  
**Tissue:** Whole bone marrow (sorted subpopulation samples excluded)  
**Repo:** github.com/glenritschel/aiha-scrna  
**Output dir:** /content/drive/MyDrive/Ritschel_Research/aiha_scrna_output

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted.")

In [ ]:
%%capture
!pip install scanpy anndata scvi-tools gseapy igraph leidenalg

In [ ]:
import os, subprocess

REPO_DIR = '/content/aiha-scrna'
if not os.path.exists(f'{REPO_DIR}/src/01_load_qc.py'):
    subprocess.run(['git', 'clone',
                    'https://github.com/glenritschel/aiha-scrna.git',
                    REPO_DIR], check=True)
    print("Repo cloned.")
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
    print("Repo up to date.")
print(f"Scripts: {sorted(os.listdir(REPO_DIR + '/src'))}")

In [ ]:
import os, gc, re, time, json, random
import numpy as np
import pandas as pd
import scipy.io
import anndata as ad
import scanpy as sc

sc.settings.verbosity = 1

DRIVE_BASE = "/content/drive/MyDrive/Ritschel_Research/aiha_scrna_output"
RAW_DIR    = os.path.join(DRIVE_BASE, "raw")
PROCESSED  = os.path.join(DRIVE_BASE, "processed")
os.makedirs(PROCESSED, exist_ok=True)
os.makedirs(RAW_DIR,   exist_ok=True)

print(f"RAW_DIR  : {RAW_DIR}")
print(f"PROCESSED: {PROCESSED}")
print(f"Files in raw: {len(os.listdir(RAW_DIR))} | processed: {len(os.listdir(PROCESSED))}")

## Data Download

In [ ]:
# Download GSE301528_RAW.tar (~663 MB) if not already present
import os

tar_path = os.path.join(RAW_DIR, "GSE301528_RAW.tar")
if not os.path.exists(tar_path) or os.path.getsize(tar_path) < 5e8:
    print("Downloading GSE301528_RAW.tar (~663 MB)...")
    !wget -q --show-progress -P {RAW_DIR} \
        "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE301nnn/GSE301528/suppl/GSE301528_RAW.tar"
    print("Extracting...")
    !tar -xf {tar_path} -C {RAW_DIR}
    print("Done.")
else:
    print("Data already present.")

mtx_files = [f for f in os.listdir(RAW_DIR) if f.endswith('_matrix.mtx.gz')]
print(f"MTX files in raw: {len(mtx_files)} (expect 18 whole-BM + 6 sorted subpops = 24)")

## Script 01 — Load & QC
Loads 12 whole-BM samples (sorted subpop samples GSM9084848–853 excluded by condition map).  
Per-condition checkpointing manages RAM. QC filter → normalise → HVG selection → save.

In [ ]:
OUT_01 = os.path.join(PROCESSED, "01_loaded.h5ad")
if os.path.exists(OUT_01):
    print("01_loaded.h5ad found — skipping Script 01.")
else:
    %run {REPO_DIR}/src/01_load_qc.py

## Script 02 — scVI Embedding & Clustering
Train scVI (200 epochs, T4 GPU recommended). Checkpoint saves immediately after training —  
if the session resets after training, the resume cell below picks up without retraining.

In [ ]:
OUT_02   = os.path.join(PROCESSED, "02_scvi.h5ad")
CKPT_02  = os.path.join(PROCESSED, "02_scvi_ckpt.h5ad")

if os.path.exists(OUT_02):
    print("02_scvi.h5ad found — skipping Script 02.")
else:
    %run {REPO_DIR}/src/02_scvi_embed.py

In [ ]:
# ── Resume from checkpoint if session reset after training ───────────────────
# Run this cell only if 02_scvi.h5ad is missing but 02_scvi_ckpt.h5ad exists.
import os
OUT_02  = os.path.join(PROCESSED, "02_scvi.h5ad")
CKPT_02 = os.path.join(PROCESSED, "02_scvi_ckpt.h5ad")

if not os.path.exists(OUT_02) and os.path.exists(CKPT_02):
    import torch, random
    RANDOM_SEED = 0
    np.random.seed(RANDOM_SEED); random.seed(RANDOM_SEED); torch.manual_seed(RANDOM_SEED)

    print("Loading scVI checkpoint — completing UMAP + Leiden...")
    adata = sc.read_h5ad(CKPT_02)
    print(f"  {adata.n_obs:,} cells | X_scVI present: {'X_scVI' in adata.obsm}")

    sc.pp.neighbors(adata, use_rep="X_scVI", n_neighbors=15)
    sc.tl.umap(adata)
    print("UMAP complete")

    for res in [0.5, 0.8, 1.2]:
        sc.tl.leiden(adata, resolution=res, key_added=f"leiden_{res}",
                     random_state=RANDOM_SEED, flavor="igraph",
                     n_iterations=2, directed=False)
        print(f"  Resolution {res}: {adata.obs[f'leiden_{res}'].nunique()} clusters")

    # Resolution selection
    best_res = 0.5   # update if a different resolution is preferred
    adata.obs["leiden"] = adata.obs[f"leiden_{best_res}"].copy()
    adata.uns["leiden_resolution"] = best_res

    adata.obs.groupby(["leiden","condition"]).size().unstack(fill_value=0).to_csv(
        os.path.join(PROCESSED, "cluster_condition_counts.csv"))

    adata.write_h5ad(OUT_02)
    print(f"Saved: {OUT_02}")
elif os.path.exists(OUT_02):
    print("02_scvi.h5ad already exists.")
else:
    print("Neither checkpoint nor final file found — run Script 02 cell above first.")

## Script 03 — Cell Type Annotation
12 bone marrow cell types: HSC/progenitor, erythroid progenitor, mature erythroid,  
myeloid progenitor, monocyte, T cell, NK cell, B cell, plasma cell,  
cytotoxic T, exhausted T, regulatory T.

In [ ]:
OUT_03 = os.path.join(PROCESSED, "03_annotated.h5ad")
if os.path.exists(OUT_03):
    print("03_annotated.h5ad found — skipping Script 03.")
else:
    %run {REPO_DIR}/src/03_annotate_clusters.py

## Script 04 — AIHA Signature Scoring
5 AIHA-specific signatures per cell:
- Erythrophagocytosis stress
- Complement / FcR activation
- T cell exhaustion & dysregulation
- Inflammatory cytokine axis
- Type I interferon response

In [ ]:
OUT_04 = os.path.join(PROCESSED, "04_scored.h5ad")
if os.path.exists(OUT_04):
    print("04_scored.h5ad found — skipping Script 04.")
else:
    %run {REPO_DIR}/src/04_signature_scoring.py

In [ ]:
# ── Fix: run this cell if Script 04 crashes on the condition scores table ────
# (scores may not persist if the script errors before saving)
import scanpy as sc, pandas as pd, os

OUT_04 = os.path.join(PROCESSED, "04_scored.h5ad")
if not os.path.exists(OUT_04):
    SIG_COLS = ["erythrophagocytosis_stress","complement_fcr_activation",
                "t_cell_exhaustion_dysregulation","inflammatory_cytokine_axis",
                "type_I_interferon_response"]
    adata = sc.read_h5ad(os.path.join(PROCESSED, "03_annotated.h5ad"))
    SIGS = {
        "erythrophagocytosis_stress": ["TFRC","HMOX1","SLC40A1","FTH1","FTL","KLF1",
                                       "GATA1","HBA1","HBB","ALAS2","EPB42","CYBRD1","HAMP","EPOR"],
        "complement_fcr_activation":  ["C1QA","C1QB","C1QC","C3","C4A","C4B","CR1",
                                       "FCGR1A","FCGR2A","FCGR2B","FCGR3A","FCGR3B","CD64","MRC1","CD16","CD32"],
        "t_cell_exhaustion_dysregulation": ["PDCD1","LAG3","TIGIT","CTLA4","TOX","ENTPD1",
                                            "CD38","GZMB","PRF1","IFNG","CD8A","KLRG1","CD244","HAVCR2"],
        "inflammatory_cytokine_axis": ["IL6","TNF","IL1B","IL10","IL2","IL17A","IFNG",
                                       "CXCL8","CXCL10","CCL2","CCL5","IL21","IL4","CSF1"],
        "type_I_interferon_response": ["ISG15","IFI44L","IFIT1","IFIT3","MX1","OAS1","OAS2",
                                       "IRF7","RSAD2","IFI6","OASL","IFITM1","IFITM3","HERC5"],
    }
    for sig, genes in SIGS.items():
        found = [g for g in genes if g in adata.var_names]
        sc.tl.score_genes(adata, found, score_name=sig) if found else adata.obs.__setitem__(sig, 0.0)
    print("\nScores by condition:")
    cs = adata.obs.groupby("condition", observed=False)[SIG_COLS].mean()
    cs.insert(0, "n_cells", adata.obs.groupby("condition", observed=False).size())
    print(cs.to_string())
    adata.write_h5ad(OUT_04)
    print("Saved: 04_scored.h5ad")
else:
    print("04_scored.h5ad already exists.")

## Script 05 — Differential Expression
- **Primary:** Diagnosis vs Remission
- **Secondary:** Relapse/Refractory vs Remission
- **Tertiary:** Diagnosis vs Relapse/Refractory
- Cluster-vs-rest (35 clusters), erythroid subset, T cell subset, pro-AIHA cluster DE

In [ ]:
OUT_05 = os.path.join(PROCESSED, "05_de.h5ad")
if os.path.exists(OUT_05):
    print("05_de.h5ad found — skipping Script 05.")
else:
    %run {REPO_DIR}/src/05_differential_expression.py

## Script 06 — LINCS L1000 Reversal Scoring
Enrichr queries across all DE comparisons and 35 cluster profiles.  
**Expect 30–50 minutes.** 41 total queries.

In [ ]:
OUT_06 = os.path.join(PROCESSED, "06_lincs.csv")
if os.path.exists(OUT_06):
    print("06_lincs.csv found — skipping Script 06.")
else:
    %run {REPO_DIR}/src/06_lincs_repurposing.py

## Script 07 — Novelty Prioritization
PubMed AIHA / hemolytic anemia / complement novelty filtering.  
Uses 90th-percentile threshold and strips directional suffixes from compound names.

In [ ]:
OUT_07 = os.path.join(PROCESSED, "priority_candidates.csv")

if os.path.exists(OUT_07):
    print("priority_candidates.csv found — skipping Script 07.")
else:
    # ── Inline novelty prioritization (handles suffix-stripping and threshold) ──
    import time, json, urllib.request, urllib.parse, pandas as pd

    PROCESSED = "/content/drive/MyDrive/Ritschel_Research/aiha_scrna_output/processed"
    SLEEP = 0.4

    MOA_REFERENCE = {
        "sar405838": "MDM2 inhibitor", "rg7112": "MDM2 inhibitor",
        "amg-232": "MDM2 inhibitor", "rg-7388": "MDM2 inhibitor",
        "r-547": "CDK inhibitor", "pha-848125": "CDK2/5 inhibitor",
        "c646": "p300/CBP inhibitor", "gw-9662": "PPARgamma antagonist",
        "kpt-330": "XPO1 inhibitor", "tak-733": "MEK1/2 inhibitor",
        "palbociclib": "CDK4/6 inhibitor", "alvocidib": "CDK1/2/4/6/9 inhibitor",
        "ql-xii-47": "MELK/FLT3 inhibitor", "wz-3105": "SRC/ABL inhibitor",
        "wz-4-145": "CDK8 inhibitor", "cgp-60474": "CDK1/2 inhibitor",
        "as-601245": "JNK inhibitor", "jnk-9l": "JNK inhibitor",
    }

    PUBMED_BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    def pubmed_count(query):
        params = urllib.parse.urlencode({"db":"pubmed","term":query,"retmode":"json","retmax":"0"})
        try:
            with urllib.request.urlopen(f"{PUBMED_BASE}?{params}", timeout=10) as r:
                return int(json.loads(r.read().decode())["esearchresult"]["count"])
        except Exception:
            return -1

    SEARCH_A = '("{c}"[Title/Abstract]) AND ("autoimmune hemolytic anemia"[Title/Abstract] OR "AIHA"[Title/Abstract])'
    SEARCH_B = '("{c}"[Title/Abstract]) AND ("hemolytic anemia"[Title/Abstract] OR "immune hemolysis"[Title/Abstract])'
    SEARCH_C = '("{c}"[Title/Abstract]) AND ("complement"[Title/Abstract] OR "Fc receptor"[Title/Abstract] OR "erythrophagocytosis"[Title/Abstract])'

    raw = pd.read_csv(f"{PROCESSED}/06_lincs.csv")
    raw["compound"] = raw["compound"].str.replace(r'\s+(Up|Down)$', '', regex=True).str.strip()
    raw = raw.groupby("compound").agg(
        max_reversal_score=("max_reversal_score","max"),
        n_queries=("n_queries","max")).reset_index()

    threshold = raw["max_reversal_score"].quantile(0.90)
    print(f"90th percentile threshold: {threshold:.1f}")
    candidates = raw[raw["max_reversal_score"] >= threshold].sort_values(
        "max_reversal_score", ascending=False).reset_index(drop=True)
    print(f"Candidates: {len(candidates)}")

    results = []
    for i, row in candidates.iterrows():
        c = row["compound"]
        print(f"[{i+1}/{len(candidates)}] {c}...", end=" ")
        ca = pubmed_count(SEARCH_A.format(c=c)); time.sleep(SLEEP)
        cb = pubmed_count(SEARCH_B.format(c=c)); time.sleep(SLEEP)
        cc = pubmed_count(SEARCH_C.format(c=c)); time.sleep(SLEEP)
        tier = "NOVEL_ALL" if ca==0 and cb==0 and cc==0 else                "NOVEL_AIHA" if ca==0 else "KNOWN"
        print(f"{tier} (AIHA:{ca}, Heme:{cb}, Comp:{cc})")
        results.append({"compound": c,
                        "moa": MOA_REFERENCE.get(c.lower(), "unknown"),
                        "novelty_tier": tier,
                        "max_reversal_score": row["max_reversal_score"],
                        "n_queries": row["n_queries"],
                        "aiha_pubs": ca, "heme_pubs": cb, "comp_pubs": cc})

    df = pd.DataFrame(results)
    MULT = {"NOVEL_ALL": 3.0, "NOVEL_AIHA": 1.5, "KNOWN": 1.0}
    df["novelty_multiplier"] = df["novelty_tier"].map(MULT)
    df["priority_score"] = (df["max_reversal_score"] * df["n_queries"] * df["novelty_multiplier"]).round(1)
    df = df.sort_values("priority_score", ascending=False).reset_index(drop=True)

    print(f"\nNovelty: {df['novelty_tier'].value_counts().to_dict()}")
    print(df.head(20)[["compound","moa","novelty_tier","max_reversal_score","n_queries","priority_score"]].to_string(index=False))

    pw = df[df["novelty_tier"]=="NOVEL_ALL"]
    df.to_csv(f"{PROCESSED}/priority_candidates.csv", index=False)
    pw.to_csv(f"{PROCESSED}/patent_watch.csv", index=False)
    print(f"\nPatent watch: {len(pw)} NOVEL_ALL compounds")
    print("Pipeline complete.")

## Results Summary

In [ ]:
import pandas as pd, os

PROCESSED = "/content/drive/MyDrive/Ritschel_Research/aiha_scrna_output/processed"

for fname, label in [("priority_candidates.csv","All candidates"),
                     ("patent_watch.csv","NOVEL_ALL patent watch")]:
    path = os.path.join(PROCESSED, fname)
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"\n{label}: {len(df)} compounds")
        cols = ["compound","moa","novelty_tier","max_reversal_score","n_queries","priority_score"]
        print(df.head(15)[cols].to_string(index=False))